In [0]:
import zipfile, os
from functools import reduce

In [0]:

bronze = "/Volumes/hmdaproj/default/bronze/"   # adjust if needed
#lar files unzip
lar_dfs = []
for year in range(2013, 2018):
    # unzip
    extract_dir = f"{bronze}unzip_lar_{year}/"
    with zipfile.ZipFile(f"{bronze}hmda_{year}_lar.zip") as z:   # adjust zip name to yours
        z.extractall(extract_dir)
    csv_name = [f for f in os.listdir(extract_dir) if f.endswith(".csv")][0]

    # read
    df = spark.read.option("header", True).csv(extract_dir + csv_name)
    lar_dfs.append(df)
    print(f"{year}: {len(df.columns)} cols, loaded")

2013: 78 cols, loaded
2014: 78 cols, loaded
2015: 78 cols, loaded
2016: 78 cols, loaded
2017: 78 cols, loaded


In [0]:

# union all lar files
lar = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), lar_dfs)
print("TOTAL LAR columns:", len(lar.columns))
print("TOTAL LAR rows:", lar.count())

TOTAL LAR columns: 78
TOTAL LAR rows: 31095061


In [0]:
# panel files unzip
panel_dfs = []
for year in range(2013, 2018):
    # unzip
    extract_dir = f"{bronze}unzip_panel_{year}/"
    with zipfile.ZipFile(f"{bronze}hmda_{year}_panel.zip") as z:   # adjust zip name to yours
        z.extractall(extract_dir)
    csv_name = [f for f in os.listdir(extract_dir) if f.endswith(".csv")][0]

    # read
    df = spark.read.option("header", True).csv(extract_dir + csv_name)
    panel_dfs.append(df)
    print(f"{year}: {len(df.columns)} cols, loaded")

2013: 21 cols, loaded
2014: 21 cols, loaded
2015: 21 cols, loaded
2016: 21 cols, loaded
2017: 21 cols, loaded


In [0]:
# union all lar files
panel = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), panel_dfs)
print("TOTAL Panel columns:", len(panel.columns))
print("TOTAL Panel rows:", panel.count())

TOTAL Panel columns: 21
TOTAL Panel rows: 33778


In [0]:
lar.printSchema()

root
 |-- as_of_year: string (nullable = true)
 |-- respondent_id: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- agency_abbr: string (nullable = true)
 |-- agency_code: string (nullable = true)
 |-- loan_type_name: string (nullable = true)
 |-- loan_type: string (nullable = true)
 |-- property_type_name: string (nullable = true)
 |-- property_type: string (nullable = true)
 |-- loan_purpose_name: string (nullable = true)
 |-- loan_purpose: string (nullable = true)
 |-- owner_occupancy_name: string (nullable = true)
 |-- owner_occupancy: string (nullable = true)
 |-- loan_amount_000s: string (nullable = true)
 |-- preapproval_name: string (nullable = true)
 |-- preapproval: string (nullable = true)
 |-- action_taken_name: string (nullable = true)
 |-- action_taken: string (nullable = true)
 |-- msamd_name: string (nullable = true)
 |-- msamd: string (nullable = true)
 |-- state_name: string (nullable = true)
 |-- state_abbr: string (nullable = true)
 |-- state

In [0]:
panel.printSchema()

root
 |-- Activity Year: string (nullable = true)
 |-- Respondent ID: string (nullable = true)
 |-- Agency Code: string (nullable = true)
 |-- Parent Respondent ID: string (nullable = true)
 |-- Parent Name (Panel): string (nullable = true)
 |-- Parent City (Panel): string (nullable = true)
 |-- Parent State (Panel): string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Assets: string (nullable = true)
 |-- Other Lender Code: string (nullable = true)
 |-- Respondent Name (Panel): string (nullable = true)
 |-- Respondent City (Panel): string (nullable = true)
 |-- Respondent State (Panel): string (nullable = true)
 |-- Top Holder RSSD ID: string (nullable = true)
 |-- Top Holder Name: string (nullable = true)
 |-- Top Holder City: string (nullable = true)
 |-- Top Holder State: string (nullable = true)
 |-- Top Holder Country: string (nullable = true)
 |-- Respondent RSSD ID: string (nullable = true)
 |-- Parent RSSD ID: string (nullable = true)
 |-- Respondent FIPS State 

# LAR unnecessary column dropping & type casting & dropping duplicates especially associated to key columns

In [0]:
from pyspark.sql import functions as F

In [0]:
# columns that are pure junk / mostly empty / not useful
junk=["application_date_indicator",
    "applicant_race_2","applicant_race_name_2","applicant_race_3","applicant_race_name_3",
    "applicant_race_4","applicant_race_name_4","applicant_race_5","applicant_race_name_5",
    "co_applicant_race_2","co_applicant_race_name_2","co_applicant_race_3","co_applicant_race_name_3",
    "co_applicant_race_4","co_applicant_race_name_4","co_applicant_race_5","co_applicant_race_name_5",
    "denial_reason_2","denial_reason_name_2","denial_reason_3","denial_reason_name_3",
    "loan_type","property_type","loan_purpose","owner_occupancy","preapproval",
    "action_taken","applicant_ethnicity","co_applicant_ethnicity",
    "applicant_race_1","co_applicant_race_1","applicant_sex","co_applicant_sex",
    "purchaser_type","denial_reason_1","hoepa_status","lien_status","msamd","edit_status"
]
lar=lar.drop(*junk)

In [0]:
type_cast_cols={
    "as_of_year": "int",
    "loan_amount_000s": "double",          # thousands, can keep double
    "applicant_income_000s": "double",     # thousands, double
    "rate_spread": "double",               # a rate, genuinely decimal
    "population": "long",                   # ← fixed: count
    "minority_population": "double",        # this is a PERCENTAGE, stays double
    "hud_median_family_income": "double",   # dollars, double ok
    "tract_to_msamd_income": "double",      # percentage, double
    "number_of_owner_occupied_units": "long",   # ← fixed: count
    "number_of_1_to_4_family_units": "long",    # ← fixed: count
}
for col, dtype in type_cast_cols.items():
    lar = lar.withColumn(col, F.col(col).cast(dtype))
lar.printSchema()

root
 |-- as_of_year: integer (nullable = true)
 |-- respondent_id: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- agency_abbr: string (nullable = true)
 |-- agency_code: string (nullable = true)
 |-- loan_type_name: string (nullable = true)
 |-- property_type_name: string (nullable = true)
 |-- loan_purpose_name: string (nullable = true)
 |-- owner_occupancy_name: string (nullable = true)
 |-- loan_amount_000s: double (nullable = true)
 |-- preapproval_name: string (nullable = true)
 |-- action_taken_name: string (nullable = true)
 |-- msamd_name: string (nullable = true)
 |-- state_name: string (nullable = true)
 |-- state_abbr: string (nullable = true)
 |-- state_code: string (nullable = true)
 |-- county_name: string (nullable = true)
 |-- county_code: string (nullable = true)
 |-- census_tract_number: string (nullable = true)
 |-- applicant_ethnicity_name: string (nullable = true)
 |-- co_applicant_ethnicity_name: string (nullable = true)
 |-- applicant_r

In [0]:
# remove exact duplicate rows
before = lar.count()
lar = lar.dropDuplicates()
print("dropped duplicates:", before - lar.count())

dropped duplicates: 972


In [0]:
# drop rows only where KEY fields are missing (NOT a blanket dropna)
lar = lar.dropna(subset=["as_of_year", "action_taken_name", "loan_amount_000s", "state_code"])
print("rows after key-null drop:", lar.count())

rows after key-null drop: 31044421


# panel typecasting & dropping duplicates & type casting

In [0]:
panel = (panel
    .withColumnRenamed("Respondent ID", "respondent_id")
    .withColumnRenamed("Agency Code", "agency_code")
    .withColumnRenamed("Activity Year", "as_of_year")
    .withColumnRenamed("Assets", "assets")
    .withColumnRenamed("Other Lender Code", "other_lender_code")
    .withColumnRenamed("Respondent Name (Panel)", "respondent_name")
    .withColumnRenamed("Parent Name (Panel)", "parent_name")
    .withColumnRenamed("Top Holder Name", "top_holder_name")
)
panel = panel.withColumn("as_of_year", F.col("as_of_year").cast("int"))
panel = panel.withColumn("assets", F.col("assets").cast("double"))
panel = panel.dropDuplicates()

# Join LAR + Panel, write Delta

In [0]:
#search access datalake storage
service_credential = dbutils.secrets.get(scope='hmdaScope',key='app-secret')

spark.conf.set("fs.azure.account.auth.type.hmdaprojstorage.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.hmdaprojstorage.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.hmdaprojstorage.dfs.core.windows.net", "df5e1b30-7a5d-46d4-a569-82706acec819")
spark.conf.set("fs.azure.account.oauth2.client.secret.hmdaprojstorage.dfs.core.windows.net", service_credential)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.hmdaprojstorage.dfs.core.windows.net", "https://login.microsoftonline.com/2ecf7998-4c57-4bc7-acbb-43cac8c31bfa/oauth2/token")

In [0]:
# join institution info onto each loan
silver = lar.join(
    panel.select("respondent_id","agency_code","as_of_year",
                 "assets","other_lender_code","respondent_name",
                 "parent_name","top_holder_name"),
    on=["respondent_id","agency_code","as_of_year"],
    how="left"
)

# write to silver container as Delta
silver.write.format("delta").mode("overwrite") \
    .save("abfss://silver@hmdaprojstorage.dfs.core.windows.net")

print("silver written:", silver.count(), "rows,", len(silver.columns), "columns")

silver written: 31044421 rows, 44 columns
